# 1.1 Python 语言快速上手 · 练习笔记

> 共 7 个练习：函数与类型注解、for 循环、类、装饰器、推导式、上下文管理器、工具注册表。
> 整理自 demo1.1 练习（2026-10-24），每个练习 = 说明 + 可运行代码。

## 1. 温度转换器

练习点：函数 + 类型注解 + docstring

In [1]:
def celsius_to_fahrenheit(c: float) -> float:
    """将摄氏度转换为华氏度。"""
    return c * 9 / 5 + 32


def fahrenheit_to_celsius(f: float) -> float:
    """将华氏度转换为摄氏度。"""
    return (f - 32) * 5 / 9


print(celsius_to_fahrenheit(0))    # 32.0
print(celsius_to_fahrenheit(100))  # 212.0
print(fahrenheit_to_celsius(32))   # 0.0
print(fahrenheit_to_celsius(212))  # 100.0

32.0
212.0
0.0
100.0


## 2. 质数判断器

练习点：for 循环 + 提前 return。题目原要求 **for-else** 语法，两种写法都放在下面对比（功能等价）。

In [ ]:
def is_prime(n: int) -> bool:
    """判断一个数是否是质数（for + return 写法）"""
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def is_prime_for_else(n: int) -> bool:
    """判断一个数是否是质数（for-else 写法）"""
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            break
    else:  # 循环未被 break 时执行
        return True
    return False


print(is_prime(13))            # True
print(is_prime_for_else(13))   # True
print(is_prime(10))            # False

## 3. 学生管理系统

练习点：类、实例方法、`max(key=...)`、f-string。定义的 `Student` 类会被练习 5 复用。

In [ ]:
class Student:
    def __init__(self, name: str, scores: dict[str, float]) -> None:
        self.name = name
        self.scores = scores

    def average_score(self) -> float:
        """计算平均分，无成绩时返回 0.0"""
        if not self.scores:
            return 0
        return sum(self.scores.values()) / len(self.scores)

    def best_subject(self) -> str | None:
        if not self.scores:
            return None
        return max(self.scores, key=self.scores.get)

    def __str__(self):
        return f'Student {self.name} with scores: {self.scores}'


stu = Student("cyd", {"a": 1, "b": 2})
print(stu.average_score())   # 1.5
print(stu.best_subject())    # b

## 4. 装饰器实战：@cache_result

练习点：闭包、装饰器、可变参数。

> 注意点：缓存键只用了 `args`，没把 `kwargs` 算进去——同名函数一旦传关键字参数会撞键。工业级做法是 `functools.lru_cache`。

In [ ]:
from typing import Callable


def cache_result(func: Callable) -> Callable:
    cache = {}

    def wrapper(*args, **kwargs):
        key = args
        if key in cache:
            print("读取缓存")
            return cache[key]
        result = func(*args, **kwargs)
        cache[key] = result
        print("写入缓存")
        return result

    return wrapper


@cache_result
def sum_all(*args) -> int:
    return sum(args)


print(sum_all(2, 3))  # 写入缓存 5
print(sum_all(2, 3))  # 读取缓存 5
print(sum_all(2, 3))  # 读取缓存 5
print(sum_all(3, 3))  # 写入缓存 6
print(sum_all(2, 3))  # 读取缓存 5
print(sum_all(3, 3))  # 读取缓存 6

## 5. 推导式转换

练习点：for 循环改写为列表推导式（依赖练习 3 的 `Student` 类，需先运行上面单元格）。

In [ ]:
students: list[Student] = [
    Student("Alice", {"math": 90, "english": 55}),
    Student("Bob", {"math": 45, "english": 70}),
    Student("Carol", {"math": 80, "english": 85}),
]

# 原始 for 循环写法：
result = []
for student in students:
    if student.average_score() >= 60:
        result.append(student.name)

# 列表推导式写法：
result = [student.name for student in students if student.average_score() >= 60]
print(result)  # ['Alice', 'Carol']

## 6. 上下文管理器实战：FileBackup

练习点：`__enter__` / `__exit__`、with 语句、shutil.copy2。

> 注意点：
> - `__exit__` 返回 `False` 表示异常照常向上抛、不吞掉——与工程规范 "fail fast" 一致。
> - `__enter__` 返回了打开的文件但 `__exit__` 没有 close()，更稳的写法是持有句柄并在退出时关闭。
> - 原练习的返回注解是 `TextIOWrapper[_WrappedBuffer]`，但 `_WrappedBuffer` 是 3.13 的私有类型（3.14 已移除），为跨版本可运行这里简写为 `TextIOWrapper`。
> - 单元格开头自建 `test.txt`（当前目录没有该文件），保证开箱可运行。

In [ ]:
from io import TextIOWrapper
from pathlib import Path
from shutil import copy2

Path("test.txt").write_text("test66666", encoding="utf-8")


class FileBackup:

    def __init__(self, filepath: str) -> None:
        self.filepath = filepath
        self.backup_path = filepath + ".bak"

    def __enter__(self) -> TextIOWrapper:
        copy2(self.filepath, self.backup_path)    # 进入时备份
        return open(self.filepath, "r")

    def __exit__(self, *args) -> bool:
        print(f"__exit__方法{self.backup_path}")  # 退出时打印备份路径
        return False


with FileBackup("test.txt") as file:
    print(file.readline())

## 7. 工具注册表

练习点：带参数的装饰器（三层闭包）、字典作为注册表。

> 这个模式正是 Agent 开发中 **Tool Registry** 的雏形——第四周学 Function Calling 时会再次见到它。

In [ ]:
from typing import Callable

tool_registry = {}


def register_tool(name: str) -> Callable:
    def decorator(func: Callable) -> Callable:
        tool_registry[name] = func
        return func
    return decorator


@register_tool("search_web")
def search_web(query: str) -> list[str]:
    """搜索网络"""
    return [f"结果: {query}"]


print(tool_registry.keys())                      # dict_keys(['search_web'])
print(tool_registry.get("search_web", None))     # <function search_web ...>
print(tool_registry.get("search_web2", None))    # None

search_web_fn = tool_registry.get("search_web", None)
if search_web_fn is not None:
    print("找到function")
    print(search_web_fn("调用search_web"))       # ['结果: 调用search_web']